<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the core libraries
!pip install -U transformers datasets evaluate accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
   

In [2]:
from transformers import pipeline

# This automatically downloads the default sentiment model and tokenizer
classifier = pipeline("sentiment-analysis")

# Test it
result = classifier("I am so excited to start Project 15!")
print(result)
# Expected Output: [{'label': 'POSITIVE', 'score': 0.999}]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9997624754905701}]


In [3]:
from google.colab import userdata
from huggingface_hub import login

# This pulls the token from your Colab Secrets safely
hf_token = userdata.get('HF_TOKEN')

# Log into the Hugging Face Hub
login(token=hf_token)

In [5]:
# Install core libraries
!pip install -U transformers datasets evaluate accelerate huggingface_hub

import os
from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Mount Google Drive for saving results
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment'
os.makedirs(PROJECT_PATH, exist_ok=True)

# 2. Login using your HF Token (Assumes you saved it in Colab Secrets as 'HF_TOKEN')
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub.")
except Exception as e:
    print("Login failed. Make sure 'HF_TOKEN' is in your Colab Secrets (key icon).")

Mounted at /content/drive
Successfully logged into Hugging Face Hub.


In [6]:
# Load the IMDB dataset
print("Downloading IMDB dataset...")
raw_datasets = load_dataset("imdb")

# Initialize the DistilBERT tokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

print(f"Tokenizer loaded: {model_ckpt}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [7]:
def tokenize_function(examples):
    # This turns text into input_ids and attention_masks
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

print("Mapping tokenization across the dataset (this may take 5 mins)...")
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Post-processing: remove raw text and prepare for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

Mapping tokenization across the dataset (this may take 5 mins)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [8]:
# Verify the shape: Expected [512] for each sample
sample_ids = tokenized_datasets["train"][0]["input_ids"]
print(f"Verification: First review token length is {len(sample_ids)}")

# Save to Disk
save_path = os.path.join(PROJECT_PATH, "tokenized_data")
tokenized_datasets.save_to_disk(save_path)
print(f"Week 1 Complete! Data saved to: {save_path}")

Verification: First review token length is 512


Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Week 1 Complete! Data saved to: /content/drive/MyDrive/distilbert-imdb-sentiment/tokenized_data
